In [1]:
import copy
import math

X = "X"
O = "O"
EMPTY = None

def initial_state():
    """Khởi tạo bàn cờ trống 3x3"""
    return [[EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY]]

def player(board, user_char, turn_count):
    """Xác định ai là người đi tiếp. Người chơi luôn đi lượt 0, 2, 4..."""
    ai_char = O if user_char == X else X
    return user_char if turn_count % 2 == 0 else ai_char

def actions(board):
    """Trả về danh sách các ô còn trống (hàng, cột)"""
    return {(i, j) for i in range(3) for j in range(3) if board[i][j] == EMPTY}

def result(board, action, p_char):
    """Thực hiện nước đi và trả về bàn cờ mới"""
    i, j = action
    new_board = copy.deepcopy(board)
    new_board[i][j] = p_char
    return new_board

def winner(board):
    """Kiểm tra xem có người thắng hay chưa"""
    for i in range(3):
        # Kiểm tra hàng ngang và dọc
        if board[i][0] == board[i][1] == board[i][2] != EMPTY: return board[i][0]
        if board[0][i] == board[1][i] == board[2][i] != EMPTY: return board[0][i]
    # Kiểm tra đường chéo
    if board[0][0] == board[1][1] == board[2][2] != EMPTY: return board[0][0]
    if board[0][2] == board[1][1] == board[2][0] != EMPTY: return board[0][2]
    return None

def terminal(board):
    """Kết thúc khi có người thắng hoặc không còn ô trống"""
    return winner(board) is not None or all(EMPTY not in row for row in board)

def utility(board, user_char):
    """Tính điểm: AI thắng = 1, Người thắng = -1, Hòa = 0"""
    res = winner(board)
    ai_char = O if user_char == X else X
    if res == ai_char: return 1
    if res == user_char: return -1
    return 0

# --- THUẬT TOÁN MINIMAX CƠ BẢN ---
def minimax(board, user_char):
    """Tìm nước đi tối ưu cho AI"""
    if terminal(board):
        return None

    ai_char = O if user_char == X else X
    best_val = -math.inf
    move = None

    # AI (Maximizer) thử mọi nước đi khả thi và chọn giá trị cao nhất
    for a in actions(board):
        # Sau nước của AI, đến lượt người chơi (Minimizer) nên gọi minValue
        val = minValue(result(board, a, ai_char), user_char)
        if val > best_val:
            best_val = val
            move = a
    return move

def maxValue(state, user_char):
    """Tìm giá trị lớn nhất mà AI có thể đạt được"""
    if terminal(state):
        return utility(state, user_char)

    v = -math.inf
    ai_char = O if user_char == X else X
    for action in actions(state):
        # Đối thủ sẽ cố gắng giảm điểm số, nên gọi minValue
        v = max(v, minValue(result(state, action, ai_char), user_char))
    return v

def minValue(state, user_char):
    """Tìm giá trị nhỏ nhất (người chơi cố gắng thắng AI)"""
    if terminal(state):
        return utility(state, user_char)

    v = math.inf
    # Sau lượt này là lượt của AI (Maximizer) nên gọi maxValue
    for action in actions(state):
        v = min(v, maxValue(result(state, action, user_char), user_char))
    return v

# --- GIAO DIỆN HIỂN THỊ ---
def print_board(board):
    """In bàn cờ kẻ bảng theo đúng yêu cầu giao diện"""
    print("\n      0   1   2")
    print("    -------------")
    for i, row in enumerate(board):
        row_str = f"  {i} |"
        for cell in row:
            char = cell if cell else "."
            row_str += f" {char} |"
        print(row_str)
        print("    -------------")

if __name__ == "__main__":
    board = initial_state()
    print("--- TRÒ CHƠI TIC-TAC-TOE AI (MINIMAX) ---")
    user_char = ""
    while user_char not in [X, O]:
        user_char = input("Bạn muốn chơi X hay O? (X/O): ").upper()

    turn = 0
    while not terminal(board):
        print_board(board)
        curr_p = player(board, user_char, turn)

        if curr_p == user_char:
            print(f"\n>> Lượt của bạn ({user_char})")
            try:
                r = int(input("Nhập hàng (0-2): "))
                c = int(input("Nhập cột (0-2): "))
                if (r, c) not in actions(board):
                    print("!!! Ô đã có người đánh hoặc không hợp lệ.")
                    continue
                board = result(board, (r, c), user_char)
            except ValueError:
                print("!!! Vui lòng nhập số từ 0 đến 2.")
                continue
        else:
            print(f"\n>> AI ({curr_p}) đang suy nghĩ...")
            move = minimax(board, user_char)
            board = result(board, move, curr_p)

        turn += 1

    print_board(board)
    res = winner(board)
    if res:
        print(f"\nKẾT THÚC: {res} CHIẾN THẮNG!")
    else:
        print("\nKẾT THÚC: HÒA CỜ!")

--- TRÒ CHƠI TIC-TAC-TOE AI (MINIMAX) ---
Bạn muốn chơi X hay O? (X/O): X

      0   1   2
    -------------
  0 | . | . | . |
    -------------
  1 | . | . | . |
    -------------
  2 | . | . | . |
    -------------

>> Lượt của bạn (X)
Nhập hàng (0-2): 0
Nhập cột (0-2): 1

      0   1   2
    -------------
  0 | . | X | . |
    -------------
  1 | . | . | . |
    -------------
  2 | . | . | . |
    -------------

>> AI (O) đang suy nghĩ...

      0   1   2
    -------------
  0 | . | X | . |
    -------------
  1 | . | . | . |
    -------------
  2 | . | O | . |
    -------------

>> Lượt của bạn (X)
Nhập hàng (0-2): 1
Nhập cột (0-2): 1

      0   1   2
    -------------
  0 | . | X | . |
    -------------
  1 | . | X | . |
    -------------
  2 | . | O | . |
    -------------

>> AI (O) đang suy nghĩ...

      0   1   2
    -------------
  0 | O | X | . |
    -------------
  1 | . | X | . |
    -------------
  2 | . | O | . |
    -------------

>> Lượt của bạn (X)
Nhập hàng (0-2): 2